In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Labs"
)

xpt_file = DATA_DIR / "PH_C.xpt"
csv_file = DATA_DIR / "PH_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep original for final fidelity comparison
source_df = df.copy()


print("=" * 70)
print("NHANES 2003-2004 PH_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED VARIABLES
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match expected PH_C structure.")
else:
    print("\nCHECK: Column structure differs.")
    print("Actual:")
    print(df.columns.tolist())


# ============================================================
# 4. KNOWN PANDAS XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny values:",
    f"{int(tiny_counts.sum()):,}"
)

print(
    "Columns containing tiny values:",
    int((tiny_counts > 0).sum())
)


if tiny_counts.sum() > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ].sort_values(
            ascending=False
        )
    )


# ============================================================
# 5. VARIABLES WHERE CDC ALLOWS ZERO
# ============================================================

#
# IMPORTANT:
#
# Only these variables have ranges/codes permitting zero.
#
# Therefore tiny-value restoration is restricted to these
# variables.
#

ZERO_ALLOWED_COLUMNS = [
    "PHACOFHR",
    "PHACOFMN",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHAANTHR",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


# ============================================================
# 6. SAFETY CHECK
# ============================================================

#
# If the tiny value appears in some OTHER variable,
# STOP instead of blindly changing it.
#

unexpected_tiny_columns = [
    col
    for col in tiny_counts.index
    if tiny_counts[col] > 0
    and col not in ZERO_ALLOWED_COLUMNS
]


if unexpected_tiny_columns:

    raise ValueError(
        "Tiny values found in variables where zero "
        "has not been approved by the codebook: "
        + str(unexpected_tiny_columns)
    )


print(
    "\nPASS: Tiny values occur only in "
    "CDC zero-permitted variables."
)


# ============================================================
# 7. RESTORE TRUE ZERO
# ============================================================

correction_report = []


for col in ZERO_ALLOWED_COLUMNS:

    if col not in df.columns:
        continue

    mask = df[col].eq(TINY_VALUE)

    count = int(mask.sum())

    if count > 0:

        correction_report.append({
            "variable": col,
            "tiny_before": count
        })

        df.loc[
            mask,
            col
        ] = 0.0


print("\n--- ZERO RESTORATION ---")

for item in correction_report:

    print(
        f"{item['variable']:<10} "
        f"{item['tiny_before']:>5,} values -> 0"
    )


# ============================================================
# 8. SEQN
# ============================================================

#
# SEQN is an identifier.
#
# 21005.0 -> 21005
#

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


# ============================================================
# 9. PH_C VALUES ARE INTEGER CODES / TIMES
# ============================================================

#
# PH_C does NOT contain decimal laboratory measurements.
#
# These fields are:
#   questionnaire codes
#   hours
#   minutes
#   examination session
#
# Therefore:
#
# 1.0 -> 1
# 2.0 -> 2
# 0.0 -> 0
#
# Missing stays blank.
#
# No numeric value is rounded unless it has first been
# verified to be a whole number.
#

for col in df.columns:

    if col == "SEQN":
        continue

    if not pd.api.types.is_numeric_dtype(df[col]):
        continue

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional.any():

        raise ValueError(
            f"{col} contains unexpected decimal values. "
            f"Values will NOT be changed."
        )

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: All PH_C code/time variables "
    "contain whole-number source values."
)


# ============================================================
# 10. CDC ROW COUNT
# ============================================================

print("\n--- CDC CODEBOOK VALIDATION ---")


if len(df) == 9179:

    print(
        "PASS: Row count = 9,179"
    )

else:

    print(
        "FAIL: Row count =",
        len(df)
    )


# ============================================================
# 11. QUESTIONNAIRE CODE FREQUENCIES
# ============================================================

EXPECTED_CODES = {

    "PHQ020": {
        1: 64,
        2: 8949
    },

    "PHQ030": {
        1: 16,
        2: 8997
    },

    "PHQ040": {
        1: 132,
        2: 8881
    },

    "PHQ050": {
        1: 13,
        2: 9000
    },

    "PHQ060": {
        1: 57,
        2: 8956
    },

    "PHDSESN": {
        0: 4297,
        1: 3400,
        2: 1482
    }
}


print("\n--- CODE FREQUENCY CHECK ---")


for col, expected in EXPECTED_CODES.items():

    actual = (
        df[col]
        .value_counts(dropna=True)
        .sort_index()
        .to_dict()
    )

    if actual == expected:

        print(
            f"PASS: {col} matches CDC."
        )

    else:

        print(
            f"FAIL: {col}"
        )

        print(
            "Expected:",
            expected
        )

        print(
            "Actual:",
            actual
        )


# ============================================================
# 12. RANGE / MISSING COUNT CHECKS
# ============================================================

#
# variable:
# (minimum, maximum, valid count, missing count)
#

EXPECTED_RANGES = {

    "PHACOFHR": (
        0, 21,
        64, 9115
    ),

    "PHACOFMN": (
        0, 59,
        64, 9115
    ),

    "PHAALCHR": (
        1, 19,
        16, 9163
    ),

    "PHAALCMN": (
        1, 57,
        16, 9163
    ),

    "PHAGUMHR": (
        0, 18,
        132, 9047
    ),

    "PHAGUMMN": (
        0, 58,
        132, 9047
    ),

    "PHAANTHR": (
        0, 12,
        13, 9166
    ),

    "PHAANTMN": (
        17, 59,
        13, 9166
    ),

    "PHASUPHR": (
        0, 15,
        57, 9122
    ),

    "PHASUPMN": (
        0, 59,
        57, 9122
    ),

    "PHAFSTHR": (
        0, 184,
        9013, 166
    ),

    "PHAFSTMN": (
        0, 59,
        9013, 166
    )
}


print("\n--- RANGE CHECK ---")


for col, (
    min_allowed,
    max_allowed,
    expected_valid,
    expected_missing
) in EXPECTED_RANGES.items():

    s = df[col]

    valid_count = int(
        s.notna().sum()
    )

    missing_count = int(
        s.isna().sum()
    )

    outside = (
        s.notna()
        &
        (
            (s < min_allowed)
            |
            (s > max_allowed)
        )
    )

    outside_count = int(
        outside.sum()
    )


    if (
        valid_count == expected_valid
        and
        missing_count == expected_missing
        and
        outside_count == 0
    ):

        print(
            f"PASS: {col:<10} "
            f"range {min_allowed}-{max_allowed}, "
            f"valid={valid_count:,}, "
            f"missing={missing_count:,}"
        )

    else:

        print(
            f"FAIL: {col}"
        )

        print(
            "Valid:",
            valid_count,
            "Expected:",
            expected_valid
        )

        print(
            "Missing:",
            missing_count,
            "Expected:",
            expected_missing
        )

        print(
            "Outside range:",
            outside_count
        )


# ============================================================
# 13. PHDSESN ZERO CHECK
# ============================================================

#
# CDC:
#
# 0 = Morning   4,297
# 1 = Afternoon 3,400
# 2 = Evening   1,482
#

print("\n--- PHDSESN ZERO VALIDATION ---")


session_zero_count = int(
    df["PHDSESN"]
    .eq(0)
    .sum()
)


print(
    "PHDSESN code 0:",
    f"{session_zero_count:,}"
)


if session_zero_count == 4297:

    print(
        "PASS: PHDSESN 0 = Morning "
        "matches CDC count 4,297."
    )

else:

    print(
        "FAIL: Expected 4,297."
    )


# ============================================================
# 14. FINAL TINY-VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero-artifact "
        "values remain."
    )

else:

    print(
        "FAIL:",
        remaining_tiny,
        "tiny values remain."
    )


# ============================================================
# 15. SEQN VALIDATION
# ============================================================

print("\n--- SEQN VALIDATION ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 17. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False
)


# ============================================================
# 18. XPT -> CSV VALUE FIDELITY
# ============================================================

#
# Build comparison copy from original XPT.
#
# The ONLY intended data correction is:
#
# tiny value -> 0
#
# SEQN / whole-number formatting does not change
# the numeric value.
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    if col in expected_df.columns:

        expected_df.loc[
            expected_df[col].eq(TINY_VALUE),
            col
        ] = 0.0


print("\n--- XPT -> CSV FIDELITY CHECK ---")


for col in expected_df.columns:

    xpt_values = (
        expected_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        xpt_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col} preserved."
        )

    else:

        print(
            f"FAIL: {col} differs from XPT."
        )


# ============================================================
# 19. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


if TINY_VALUE.__str__() not in csv_text:

    print(
        "PASS: Tiny artifact not written to CSV."
    )


# Since every PH_C value is integral,
# no ".0" values should be needed.

dot_zero_count = csv_text.count(".0")


print(
    "Occurrences of '.0' in CSV:",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    print(
        "CHECK: CSV contains .0 formatting."
    )


print("\n" + "=" * 70)
print("PH_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 PH_C XPT -> CSV
Rows: 9,179
Columns: 19

Variables:
['SEQN', 'PHQ020', 'PHACOFHR', 'PHACOFMN', 'PHQ030', 'PHAALCHR', 'PHAALCMN', 'PHQ040', 'PHAGUMHR', 'PHAGUMMN', 'PHQ050', 'PHAANTHR', 'PHAANTMN', 'PHQ060', 'PHASUPHR', 'PHASUPMN', 'PHAFSTHR', 'PHAFSTMN', 'PHDSESN']

PASS: Column names/order match expected PH_C structure.

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny values: 4,956
Columns containing tiny values: 10

Tiny values by variable:
PHDSESN     4297
PHAFSTHR     422
PHAFSTMN     186
PHAGUMHR      34
PHAGUMMN       5
PHACOFHR       4
PHASUPHR       4
PHASUPMN       2
PHACOFMN       1
PHAANTHR       1
dtype: int64

PASS: Tiny values occur only in CDC zero-permitted variables.

--- ZERO RESTORATION ---
PHACOFHR       4 values -> 0
PHACOFMN       1 values -> 0
PHAGUMHR      34 values -> 0
PHAGUMMN       5 values -> 0
PHAANTHR       1 values -> 0
PHASUPHR       4 values -> 0
PHASUPMN       2 values -> 0
PHAFSTHR     422 values -> 0
PHAFSTMN     186 values -> 